# Task 1: repository and Kaggle setup

Run on Kaggle with **Accelerator: GPU T4 x2** and **Internet: on**. The notebook clones the project,
installs dependencies without touching Kaggle's preinstalled CUDA PyTorch, checks that two T4 GPUs are
visible, prints package versions and runs the public adapter tests. Everything is written under
`/kaggle/working` so a saved notebook version keeps it.

In [ ]:
REPO_URL = "https://github.com/AyanAnees326/small-model-alignment.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/small-model-alignment"

!git clone --depth 1 --branch {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log -1 --oneline

Install exactly as the manual recommends (§5.2): no PyTorch reinstall, drop the stale `torchao`, install the repo with `--no-deps`. `trl`/`peft` are only needed for Task 5.

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!python -m pip install -q "trl>=0.19" peft --no-deps

In [ ]:
!nvidia-smi

In [ ]:
import torch, transformers, datasets

print("torch       ", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("GPUs        ", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "expected the GPU T4 x2 accelerator"

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py tests/test_core_edge_cases.py

## Optional: full pipeline on Kaggle

The submitted runs were trained locally (see the report, §14.3). To reproduce them on T4 x2 instead, run
the stages below; `alignment.train` puts the policy on `cuda:0` and the frozen reference on `cuda:1`
automatically, and `--resume` continues from `<out>/last/state.pt` after a disconnect.

```bash
python -m alignment.data                                   # data/processed/*.jsonl + outputs/data_manifest.json
bash scripts/run_local_pipeline.sh                         # SFT -> generations -> 2x DPO -> LoRA-DPO -> pairs -> plots
```